In [2]:
# Requires timm — install with: pip install timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 88.0 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-curand-cu12
    Found existing installation: nvidia-curand-cu12 10.3.9.90
    Uninstalling nvidia-curand-cu12-1

In [3]:
import os, random, shutil
from glob import glob
import torch
import torch.nn.functional as F
from torch import nn, optim, Tensor
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import timm
import pandas as pd

In [4]:
# reproducibility
random.seed(42)
torch.manual_seed(42)
USE_VIT = True
IMG_SIZE = 224 if USE_VIT else 256

In [5]:
# 1) Stratified sampling + split
def stratified_sample_split(raw_dir, out_base, total,
                            splits=('train','val','test'),
                            ratios=(0.8,0.1,0.1)):
    classes = [d for d in os.listdir(raw_dir) if os.path.isdir(os.path.join(raw_dir,d))]
    per_cls = total // len(classes)
    for cls in classes:
        imgs = glob(os.path.join(raw_dir,cls,'*'))
        sel  = imgs if len(imgs) <= per_cls else random.sample(imgs, per_cls)
        random.shuffle(sel)
        n     = len(sel)
        i1    = int(ratios[0] * n)
        i2    = i1 + int(ratios[1] * n)
        groups = {'train': sel[:i1], 'val': sel[i1:i2], 'test': sel[i2:]}
        for split, files in groups.items():
            d = os.path.join(out_base, split, cls)
            os.makedirs(d, exist_ok=True)
            for f in files:
                shutil.copy(f, d)

In [6]:
# Dataset roots — set env vars or place datasets under ./data/
# CONTENT_DIR: 'image-classification' Kaggle dataset (images/images)
# STYLE_DIR:   'wikiart' Kaggle dataset
RAW_CONTENT = os.environ.get('CONTENT_DIR', 'data/image-classification/images/images')
RAW_STYLE   = os.environ.get('STYLE_DIR',   'data/wikiart')
WORK_BASE   = os.environ.get('WORK_DIR',    'outputs')
DATA_BASE   = os.path.join(WORK_BASE, 'data')
# Style weight lambda (paper uses ~10 with multi-layer mean/std losses)
STYLE_WEIGHT = float(os.environ.get('STYLE_WEIGHT', 10.0))


In [7]:
# sample 7k content, 3k style
stratified_sample_split(RAW_CONTENT, os.path.join(DATA_BASE,'content'), total=7000)
stratified_sample_split(RAW_STYLE,   os.path.join(DATA_BASE,'style'),   total=3000)

print('Data directories created under', DATA_BASE)

Data directories created under /kaggle/working/data


In [8]:
# 2) Print counts
print("Counts per split:")
for kind in ('content','style'):
    for split in ('train','val','test'):
        path = os.path.join(DATA_BASE, kind, split)
        cnt = sum(len(f) for _,_,f in os.walk(path))
        print(f"  {kind:7s} {split:5s}: {cnt}")


Counts per split:
  content train: 5600
  content val  : 700
  content test : 700
  style   train: 2366
  style   val  : 295
  style   test : 322


In [9]:
# 3) Transforms & Dataset
tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])


In [10]:
class StyleDS(Dataset):
    def __init__(self, root, split):
        self.C = glob(os.path.join(root,'content',split,'*','*'))
        self.S = glob(os.path.join(root,'style',  split,'*','*'))
    def __len__(self):
        return min(len(self.C), len(self.S))
    def __getitem__(self, i):
        c = Image.open(self.C[i]).convert('RGB')
        s = Image.open(self.S[i % len(self.S)]).convert('RGB')
        return tf(c), tf(s), torch.rand(1)

In [11]:
train_dl = DataLoader(StyleDS(DATA_BASE,'train'), batch_size=8, shuffle=True, num_workers=2)
val_dl   = DataLoader(StyleDS(DATA_BASE,'val'),   batch_size=8, num_workers=2)
test_dl  = DataLoader(StyleDS(DATA_BASE,'test'),  batch_size=4, num_workers=2)


In [12]:
def adain(content_feat: Tensor, style_feat: Tensor, eps: float = 1e-5) -> Tensor:
    c_mean = content_feat.mean(dim=[2,3], keepdim=True)
    c_std  = content_feat.std(dim=[2,3], keepdim=True)
    s_mean = style_feat.mean(dim=[2,3], keepdim=True)
    s_std  = style_feat.std(dim=[2,3], keepdim=True)
    normalized = (content_feat - c_mean) / (c_std + eps)
    return normalized * s_std + s_mean

In [13]:
class StyleNet(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(pretrained=True).features
        self.enc = nn.Sequential(*list(vgg)[:21])
        for p in self.enc.parameters():
            p.requires_grad=False
        self.dec = nn.Sequential(
            nn.Conv2d(512,256,3,1,1), nn.ReLU(), nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(256,256,3,1,1), nn.ReLU(),
            nn.Conv2d(256,128,3,1,1), nn.ReLU(), nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(128,64,3,1,1),  nn.ReLU(), nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(64,3,3,1,1), nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        )
    def forward(self, c, s, t):
        cf = self.enc(c)
        sf = self.enc(s)
        tf_ = adain(cf, sf)
        feat = t.view(-1,1,1,1)*tf_ + (1-t).view(-1,1,1,1)*cf
        return self.dec(feat), feat


# To use the ViT variant, uncomment below:
class ViTStyleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True, features_only=True)
        for p in self.vit.parameters():
            p.requires_grad=False
        self.dec = nn.Sequential(
            nn.Conv2d(768,256,3,1,1), nn.ReLU(), nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(256,128,3,1,1), nn.ReLU(), nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(128,64,3,1,1),  nn.ReLU(), nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(64,3,3,1,1),     nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        )
    def forward(self, c, s, t):
        cf = self.vit(c)[-1]
        sf = self.vit(s)[-1]
        tf_ = adain(cf, sf)
        feat = t.view(-1,1,1,1)*tf_ + (1-t).view(-1,1,1,1)*cf
        return self.dec(feat), feat



In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if USE_VIT:
    model = ViTStyleNet().to(device)
    # disable strict image size check and adjust for 256x256 if needed
    # pe = model.vit.patch_embed
    # pe.strict_img_size = False
    # pe.img_size = (IMG_SIZE, IMG_SIZE)
    # pe.grid_size = (IMG_SIZE//pe.patch_size, IMG_SIZE//pe.patch_size)
    # pe.num_patches = pe.grid_size[0] * pe.grid_size[1]
else:
    model = StyleNet().to(device)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [15]:
mse      = nn.MSELoss()
opt      = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
vgg_loss = models.vgg19(pretrained=True).features[:21].to(device).eval()

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:02<00:00, 216MB/s] 


In [16]:
# Multi-layer VGG feature taps (relu1_1, relu2_1, relu3_1, relu4_1)
def vgg_taps(x: Tensor):
    taps = []
    for i, layer in enumerate(vgg_loss):
        x = layer(x)
        if i in (1, 6, 11, 20):
            taps.append(x)
    return taps

# Paper's style loss: match channel mean AND std at each tapped layer
def style_loss(o_taps, s_taps) -> Tensor:
    l = 0.0
    for of, sf in zip(o_taps, s_taps):
        l += mse(of.mean(dim=[2,3]), sf.mean(dim=[2,3]))
        l += mse(of.std(dim=[2,3]),  sf.std(dim=[2,3]))
    return l

In [17]:
def loss_fn(o: Tensor, feat_mix: Tensor, s: Tensor) -> Tensor:
    o = F.interpolate(o, size=(IMG_SIZE,IMG_SIZE), mode='bilinear', align_corners=False)
    s = F.interpolate(s, size=(IMG_SIZE,IMG_SIZE), mode='bilinear', align_corners=False)
    o_taps = vgg_taps(o)
    s_taps = vgg_taps(s)
    content = mse(F.interpolate(o_taps[3], size=feat_mix.shape[2:], mode='bilinear', align_corners=False), feat_mix)
    style   = style_loss(o_taps, s_taps)
    return content + STYLE_WEIGHT * style

In [ ]:
best_val, patience, counter = float('inf'), 5, 0
for ep in range(1, 51):
    model.train()
    tloss = 0.0
    for c,s,_ in train_dl:
        c,s = c.to(device), s.to(device)
        t = torch.ones(c.size(0),1,device=device)  # train at full stylization (paper); t is a test-time dial
        opt.zero_grad()
        out, feat_mix = model(c,s,t)
        l   = loss_fn(out, feat_mix, s)
        l.backward()
        opt.step()
        tloss += l.item()
    tloss /= len(train_dl)

    model.eval()
    vloss = 0.0
    with torch.no_grad():
        for c,s,_ in val_dl:
            c,s = c.to(device), s.to(device)
            t_t  = torch.ones(c.size(0),1,device=device)
            out, feat_mix = model(c,s,t_t)
            vloss += loss_fn(out, feat_mix, s).item()
    vloss /= len(val_dl)

    print(f"Epoch {ep:02d} train {tloss:.4f} val {vloss:.4f}")
    if vloss < best_val:
        best_val, counter = vloss, 0
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {ep}")
            break

In [ ]:
# 6) Hyperparameter sweep over t on train/val/test splits
def compute_losses(output, feat_mix, style):
    o = F.interpolate(output, size=(IMG_SIZE,IMG_SIZE), mode='bilinear', align_corners=False)
    s = F.interpolate(style,  size=(IMG_SIZE,IMG_SIZE), mode='bilinear', align_corners=False)
    o_taps = vgg_taps(o)
    s_taps = vgg_taps(s)
    cl = mse(F.interpolate(o_taps[3], size=feat_mix.shape[2:], mode='bilinear', align_corners=False), feat_mix).item()
    sl = style_loss(o_taps, s_taps).item()
    return cl, sl

In [ ]:
results = []
for split_name, dl in [('train', train_dl), ('val', val_dl), ('test', test_dl)]:
    with torch.no_grad():
        for t_val in [i/10 for i in range(11)]:
            sum_cl, sum_sl, n = 0.0, 0.0, 0
            for bi,(c,s,_) in enumerate(dl):
                if bi >= 25: break  # cap sweep cost
                c,s = c.to(device), s.to(device)
                t_tensor = torch.full((c.size(0),1), t_val, device=device)
                out, feat_mix = model(c,s,t_tensor)
                cl, sl = compute_losses(out, feat_mix, s)
                sum_cl += cl
                sum_sl += sl
                n     += 1
            results.append({
                'split': split_name,
                't':          t_val,
                'content_loss': sum_cl / n,
                'style_loss':   sum_sl / n,
                'total_loss':  (sum_cl + STYLE_WEIGHT * sum_sl) / n
            })

In [ ]:
df = pd.DataFrame(results)
print(df)
best = df[df.split=='val'].sort_values('total_loss').iloc[0]
print(f"Best t on val: {best.t:.1f} (loss {best.total_loss:.4f})")

In [ ]:
# 7) Save model weights and traced module
torch.save(model.state_dict(), 'style_transfer_weights.pth')
example_c = torch.randn(1,3,IMG_SIZE,IMG_SIZE, device=device)
example_s = torch.randn(1,3,IMG_SIZE,IMG_SIZE, device=device)
example_t = torch.full((1,1), best.t, device=device)
traced = torch.jit.trace(model, (example_c, example_s, example_t))
traced.save('style_transfer_traced.pt')

In [ ]:
# 8) Test inference + collect examples
allC, allS, allO = [],[],[]
model.eval()
with torch.no_grad():
    for c,s,_ in test_dl:
        c,s = c.to(device), s.to(device)
        t_tensor = torch.full((c.size(0),1), best.t, device=device)
        out = model(c,s,t_tensor)
        for i in range(c.size(0)):
            allC.append(c[i].cpu())
            allS.append(s[i].cpu())
            allO.append(out[i].cpu())
print(f"Collected {len(allC)} samples")

In [ ]:
# 9) Plot random examples
def to_img(x):
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    im = x * std + mean
    return im.clamp(0,1).permute(1,2,0).numpy()

idxs = random.sample(range(len(allC)), 10)
fig, ax = plt.subplots(10,3,figsize=(9,30))
for row,i in enumerate(idxs):
    ax[row,0].imshow(to_img(allC[i])); ax[row,0].set_title("Content"); ax[row,0].axis('off')
    ax[row,1].imshow(to_img(allS[i])); ax[row,1].set_title("Style");   ax[row,1].axis('off')
    ax[row,2].imshow(to_img(allO[i])); ax[row,2].set_title("Output");  ax[row,2].axis('off')
plt.tight_layout()
plt.show()
